# 🎬 ClipSpark AI Engine - Development Notebook

**Architecture:** Scene-First, Stateless Signal Processing  
**Purpose:** Step-by-step implementation and verification of the video processing pipeline

---

## 📦 Setup & Dependencies

In [1]:
# Install required packages (run once)
# !pip install opencv-python numpy scenedetect[opencv] ultralytics tqdm

In [2]:
# === FFmpeg Verification & PATH Fix (Windows) ===
import subprocess
import os
import sys

def refresh_windows_path():
    if sys.platform != 'win32':
        return
    try:
        import winreg
        with winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r'SYSTEM\CurrentControlSet\Control\Session Manager\Environment') as key:
            system_path, _ = winreg.QueryValueEx(key, 'Path')
        with winreg.OpenKey(winreg.HKEY_CURRENT_USER, r'Environment') as key:
            user_path, _ = winreg.QueryValueEx(key, 'Path')
        os.environ['PATH'] = f"{system_path};{user_path}"
        print("✅ PATH refreshed")
    except Exception as e:
        print(f"⚠️ Could not refresh PATH: {e}")

def check_ffmpeg():
    try:
        result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            print(f"✅ FFmpeg found: {result.stdout.split(chr(10))[0]}")
            return True
    except:
        pass
    return False

if not check_ffmpeg():
    refresh_windows_path()
    if not check_ffmpeg():
        print("❌ FFmpeg not found. Install with: winget install ffmpeg")

✅ PATH refreshed
✅ FFmpeg found: ffmpeg version 8.0.1-full_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers


In [3]:
import os
import json
import subprocess
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Tuple, Optional, Dict, Any
from tqdm import tqdm

import cv2
import numpy as np

# ============================================================
# DOCKER-READY PATHS (Fix #3)
# Use environment variable for deployment, fallback for local dev
# ============================================================
PROJECT_ROOT = Path(os.getenv("APP_DIR", r"D:\Freelancing_Work\ClipSpark"))
TEST_VIDEOS_DIR = PROJECT_ROOT / "test_videos"
OUTPUT_DIR = PROJECT_ROOT / "output"
CLIPS_DIR = OUTPUT_DIR / "clips"

# Ensure directories exist
TEST_VIDEOS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
CLIPS_DIR.mkdir(exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Output Dir: {OUTPUT_DIR}")
print(f"Clips Dir: {CLIPS_DIR}")

Project Root: D:\Freelancing_Work\ClipSpark
Output Dir: D:\Freelancing_Work\ClipSpark\output
Clips Dir: D:\Freelancing_Work\ClipSpark\output\clips


---
## 🔧 PHASE 1: Ingestion & Optimization Layer

In [4]:
@dataclass
class VideoMetadata:
    filepath: str
    duration: float
    width: int
    height: int
    fps: float
    frame_count: int
    codec: str
    filesize_mb: float

def get_video_metadata(video_path: str) -> VideoMetadata:
    path = Path(video_path)
    if not path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")
    try:
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        fc = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))
        codec = "".join([chr((fourcc >> 8*i) & 0xFF) for i in range(4)])
        return VideoMetadata(str(path.absolute()), fc/fps if fps>0 else 0, w, h, fps, fc, codec, path.stat().st_size/(1024*1024))
    finally:
        cap.release()

VALID_EXTENSIONS = {".mp4", ".mov"}
MIN_DURATION, MAX_DURATION = 60, 300

@dataclass
class ValidationResult:
    is_valid: bool
    errors: List[str] = field(default_factory=list)
    warnings: List[str] = field(default_factory=list)
    metadata: Optional[VideoMetadata] = None

def validate_video(video_path: str) -> ValidationResult:
    errors, warnings = [], []
    path = Path(video_path)
    if not path.exists():
        return ValidationResult(False, [f"File not found: {video_path}"])
    if path.suffix.lower() not in VALID_EXTENSIONS:
        errors.append(f"Invalid format: {path.suffix}")
    try:
        metadata = get_video_metadata(video_path)
    except Exception as e:
        return ValidationResult(False, [f"Cannot read: {e}"])
    if metadata.duration < MIN_DURATION:
        errors.append(f"Too short: {metadata.duration:.1f}s < {MIN_DURATION}s")
    elif metadata.duration > MAX_DURATION:
        errors.append(f"Too long: {metadata.duration:.1f}s > {MAX_DURATION}s")
    return ValidationResult(len(errors)==0, errors, warnings, metadata)

PROXY_HEIGHT, PROXY_FPS = 360, 10

def generate_proxy_video(input_path: str, output_dir: str) -> str:
    inp = Path(input_path)
    out = Path(output_dir) / f"{inp.stem}_proxy_{PROXY_HEIGHT}p_{PROXY_FPS}fps.mp4"
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    cmd = ["ffmpeg", "-y", "-i", str(inp), "-vf", f"scale=-2:{PROXY_HEIGHT}", "-r", str(PROXY_FPS), "-an", "-c:v", "libx264", "-preset", "fast", "-crf", "23", str(out)]
    print(f"Generating proxy: {out.name}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"FFmpeg failed: {result.stderr[:500]}")
    print(f"✅ Proxy generated")
    return str(out)

print("✅ Phase 1 functions defined")

✅ Phase 1 functions defined


In [5]:
# === Run Phase 1 ===
test_videos = list(TEST_VIDEOS_DIR.glob("*.mp4")) + list(TEST_VIDEOS_DIR.glob("*.mov"))
if test_videos:
    TEST_VIDEO_PATH = str(test_videos[0])
    print(f"🎬 Using: {Path(TEST_VIDEO_PATH).name}")
    validation = validate_video(TEST_VIDEO_PATH)
    if validation.is_valid:
        print(f"✅ Valid: {validation.metadata.duration:.1f}s, {validation.metadata.width}x{validation.metadata.height}")
        proxy_path = generate_proxy_video(TEST_VIDEO_PATH, str(OUTPUT_DIR))
    else:
        print(f"❌ Invalid: {validation.errors}")
else:
    print("⚠️ No test videos found in test_videos/")

🎬 Using: testing.mp4
✅ Valid: 180.7s, 1920x1080
Generating proxy: testing_proxy_360p_10fps.mp4
✅ Proxy generated


---
## 🎬 PHASE 2: Scene Detection

In [6]:
from scenedetect import open_video, SceneManager
from scenedetect.detectors import ContentDetector

@dataclass
class Scene:
    id: int
    start_time: float
    end_time: float
    duration: float
    start_frame: int
    end_frame: int
    score: float = 0.0
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

SCENE_THRESHOLD = 27.0

def detect_scenes(video_path: str, threshold: float = SCENE_THRESHOLD) -> List[Scene]:
    print(f"Detecting scenes in: {Path(video_path).name}")
    video = open_video(video_path)
    sm = SceneManager()
    sm.add_detector(ContentDetector(threshold=threshold, min_scene_len=int(video.frame_rate)))
    sm.detect_scenes(video)
    scenes = [Scene(i+1, s.get_seconds(), e.get_seconds(), e.get_seconds()-s.get_seconds(), s.get_frames(), e.get_frames())
              for i, (s, e) in enumerate(sm.get_scene_list())]
    print(f"✅ Detected {len(scenes)} scenes")
    return scenes

print("✅ Phase 2 functions defined")

✅ Phase 2 functions defined


d:\Freelancing_Work\ClipSpark\virenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# === Run Phase 2 ===
if 'proxy_path' in dir():
    scenes = detect_scenes(proxy_path)
    print(f"\nScene durations: {[f'{s.duration:.1f}s' for s in scenes[:10]]}{'...' if len(scenes)>10 else ''}")
else:
    print("⚠️ Run Phase 1 first")

Detecting scenes in: testing_proxy_360p_10fps.mp4
✅ Detected 41 scenes

Scene durations: ['1.0s', '4.2s', '15.7s', '6.0s', '7.8s', '2.7s', '2.4s', '2.8s', '18.8s', '2.6s']...


---
## 🧠 PHASE 3: Intelligence Extraction

In [8]:
@dataclass
class MotionSignal:
    raw_scores: np.ndarray
    smoothed_scores: np.ndarray
    fps: float
    duration: float

def analyze_motion(video_path: str, smoothing_window: float = 1.0) -> MotionSignal:
    print(f"🔍 Analyzing motion...")
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    ret, prev = cap.read()
    prev_gray = cv2.GaussianBlur(cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY), (21,21), 0)
    
    raw = []
    for _ in tqdm(range(total-1), desc="Motion", unit="frame"):
        ret, curr = cap.read()
        if not ret: break
        curr_gray = cv2.GaussianBlur(cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY), (21,21), 0)
        raw.append(np.sum(cv2.absdiff(prev_gray, curr_gray)))
        prev_gray = curr_gray
    cap.release()
    
    raw = np.array(raw, dtype=np.float32)
    if raw.max() > raw.min():
        raw = (raw - raw.min()) / (raw.max() - raw.min())
    
    kernel = np.ones(int(smoothing_window * fps)) / int(smoothing_window * fps)
    smoothed = np.convolve(raw, kernel, mode='same')
    
    fps_int = int(fps)
    per_sec = np.array([smoothed[i*fps_int:(i+1)*fps_int].mean() for i in range(int(np.ceil(len(smoothed)/fps_int)))])
    
    print(f"✅ Motion: {len(per_sec)} seconds, range {per_sec.min():.2f}-{per_sec.max():.2f}")
    return MotionSignal(raw, per_sec, fps, total/fps)

print("✅ analyze_motion() defined")

✅ analyze_motion() defined


In [9]:
from ultralytics import YOLO
print("Loading YOLOv8n...")
yolo_model = YOLO('yolov8n.pt')
print("✅ YOLOv8n loaded")

Loading YOLOv8n...
✅ YOLOv8n loaded


In [10]:
@dataclass
class HumanSignal:
    raw_scores: np.ndarray
    smoothed_scores: np.ndarray
    person_counts: np.ndarray
    fps: float
    duration: float

def analyze_humans(video_path: str, sample_interval: int = 5) -> HumanSignal:
    print(f"👤 Analyzing humans...")
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_area = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) * int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    raw, counts, indices = [], [], []
    
    for fidx in tqdm(range(0, total, sample_interval), desc="Humans", unit="frame"):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
        ret, frame = cap.read()
        if not ret: break
        
        results = yolo_model(frame, verbose=False, conf=0.3)
        boxes = results[0].boxes
        person_boxes = boxes.xyxy[boxes.cls == 0].cpu().numpy()
        
        num_people = len(person_boxes)
        area = sum((b[2]-b[0])*(b[3]-b[1]) for b in person_boxes) / frame_area if frame_area > 0 else 0
        score = num_people * (1 + area)
        
        raw.append(score)
        counts.append(num_people)
        indices.append(fidx)
    
    cap.release()
    
    raw = np.array(raw, dtype=np.float32)
    if raw.max() > raw.min():
        raw = (raw - raw.min()) / (raw.max() - raw.min())
    
    duration = total / fps
    num_sec = int(np.ceil(duration))
    per_sec = np.zeros(num_sec)
    per_count = np.zeros(num_sec)
    
    for i, fidx in enumerate(indices):
        sec = int(fidx / fps)
        if sec < num_sec:
            per_sec[sec] = max(per_sec[sec], raw[i])
            per_count[sec] = max(per_count[sec], counts[i])
    
    print(f"✅ Humans: {len(per_sec)}s, max people: {int(per_count.max())}")
    return HumanSignal(raw, per_sec, per_count.astype(np.int32), fps, duration)

print("✅ analyze_humans() defined")

✅ analyze_humans() defined


In [11]:
# === Run Phase 3 ===
if 'proxy_path' in dir():
    motion_signal = analyze_motion(proxy_path)
    human_signal = analyze_humans(proxy_path)
else:
    print("⚠️ Run Phase 1 first")

🔍 Analyzing motion...


Motion: 100%|██████████| 1807/1807 [00:02<00:00, 621.42frame/s]


✅ Motion: 181 seconds, range 0.00-0.42
👤 Analyzing humans...


Humans: 100%|██████████| 362/362 [00:25<00:00, 14.47frame/s]

✅ Humans: 181s, max people: 13


---
## 🧮 PHASE 4: Scoring & Fusion Layer

**Formula:** `FusedScore(t) = 0.6 × Motion(t) + 0.4 × Human(t)`

**Rationale:** "ClipSpark" emphasizes action/intensity. Motion-heavy weighting ensures high-action moments (scoring goals, dancing) rank above static human presence.

In [12]:
# ============================================================
# SIGNAL WEIGHTS (Fix #2)
# 0.6 Motion / 0.4 Human - favors action over static presence
# ============================================================
DEFAULT_MOTION_WEIGHT = 0.6
DEFAULT_HUMAN_WEIGHT = 0.4


@dataclass
class FusedSignal:
    """Container for fused motion + human signal."""
    per_second_scores: np.ndarray
    motion_weight: float
    human_weight: float
    duration: float


def fuse_signals(
    motion: MotionSignal,
    human: HumanSignal,
    motion_weight: float = DEFAULT_MOTION_WEIGHT,
    human_weight: float = DEFAULT_HUMAN_WEIGHT
) -> FusedSignal:
    """
    Fuse motion and human signals into a single score.
    
    If human signal is all zeros (no humans detected),
    automatically shifts to 100% motion weight.
    """
    print(f"\n🧮 Fusing signals (motion: {motion_weight}, human: {human_weight})")
    
    min_len = min(len(motion.smoothed_scores), len(human.smoothed_scores))
    m_scores = motion.smoothed_scores[:min_len]
    h_scores = human.smoothed_scores[:min_len]
    
    # Handle edge case: no humans detected
    if h_scores.max() == 0:
        print("   ⚠️ No humans detected - using 100% motion")
        motion_weight = 1.0
        human_weight = 0.0
    
    fused = motion_weight * m_scores + human_weight * h_scores
    
    if fused.max() > fused.min():
        fused = (fused - fused.min()) / (fused.max() - fused.min())
    
    print(f"   ✅ Fused signal: {len(fused)} seconds")
    print(f"   Range: {fused.min():.3f} - {fused.max():.3f}")
    print(f"   Mean: {fused.mean():.3f}")
    
    return FusedSignal(fused, motion_weight, human_weight, min_len)


def score_scenes(scenes: List[Scene], fused: FusedSignal) -> List[Scene]:
    """Calculate average fused score for each scene."""
    print(f"\n📊 Scoring {len(scenes)} scenes...")
    
    scores = fused.per_second_scores
    
    for scene in scenes:
        start_sec = int(scene.start_time)
        end_sec = min(int(scene.end_time), len(scores))
        
        if start_sec < end_sec and start_sec < len(scores):
            scene.score = float(np.mean(scores[start_sec:end_sec]))
        else:
            scene.score = 0.0
    
    sorted_scenes = sorted(scenes, key=lambda s: s.score, reverse=True)
    
    print(f"\n🏆 Top 5 scenes by score:")
    for s in sorted_scenes[:5]:
        print(f"   Scene {s.id}: {s.start_time:.1f}s-{s.end_time:.1f}s ({s.duration:.1f}s) → Score: {s.score:.3f}")
    
    return scenes


print("✅ fuse_signals() and score_scenes() defined")
print(f"   Default weights: Motion={DEFAULT_MOTION_WEIGHT}, Human={DEFAULT_HUMAN_WEIGHT}")

✅ fuse_signals() and score_scenes() defined
   Default weights: Motion=0.6, Human=0.4


In [13]:
# === Run Phase 4 ===
if 'motion_signal' in dir() and 'human_signal' in dir():
    fused_signal = fuse_signals(motion_signal, human_signal)
    scenes = score_scenes(scenes, fused_signal)
else:
    print("⚠️ Run Phase 3 first")


🧮 Fusing signals (motion: 0.6, human: 0.4)
   ✅ Fused signal: 181 seconds
   Range: 0.000 - 1.000
   Mean: 0.279

📊 Scoring 41 scenes...

🏆 Top 5 scenes by score:
   Scene 39: 173.1s-174.1s (1.0s) → Score: 0.851
   Scene 38: 169.9s-173.1s (3.2s) → Score: 0.729
   Scene 37: 167.9s-169.9s (2.0s) → Score: 0.658
   Scene 32: 143.8s-149.8s (6.0s) → Score: 0.569
   Scene 25: 112.2s-117.3s (5.1s) → Score: 0.545


In [14]:
# === Visualize Fused Signal ===
def plot_fused_signal(fused: FusedSignal, scenes: List[Scene], width: int = 70) -> None:
    scores = fused.per_second_scores
    step = max(1, len(scores) // width)
    resampled = [scores[i*step:min((i+1)*step, len(scores))].mean() for i in range(min(width, len(scores)))]
    
    chars = [' ', '▁', '▂', '▃', '▄', '▅', '▆', '▇', '█']
    
    print("\n📊 FUSED INTEREST SIGNAL")
    print("=" * 75)
    print(f"1.0 |{''.join([chars[min(8, int(s * 8))] for s in resampled])}")
    print(f"    |{'─' * len(resampled)}")
    print(f"0.0 |{' ' * len(resampled)}")
    print(f"    0s{' ' * (len(resampled) - 10)}{fused.duration:.0f}s")
    
    boundary_line = [' '] * len(resampled)
    for s in scenes:
        pos = int((s.start_time / fused.duration) * len(resampled))
        if 0 <= pos < len(resampled):
            boundary_line[pos] = '|'
    print(f"    {''.join(boundary_line)} (scene boundaries)")

if 'fused_signal' in dir():
    plot_fused_signal(fused_signal, scenes)


📊 FUSED INTEREST SIGNAL
1.0 |▁▂▁▁▁▁▁▁▁ ▁  ▁ ▁ ▃▁ ▁▁  ▁▂▃▂▃▁▁▄▃▂▁▂▂▂▁▂▂▂▂▂▃▂▂▂▂▂▃▃▂▁ ▂▄▄▄▂▂▃▁   ▁▁  
    |──────────────────────────────────────────────────────────────────────
0.0 |                                                                      
    0s                                                            181s
    | |     | |  ||||      ||| ||   | || ||| ||| ||| |  || | |   ||||||||  (scene boundaries)


---
## 🎯 PHASE 5: Selection Layer (Refined Greedy Strategy)

**3-Tier Strategy:**
1. **Tier 1**: Scene fits target duration → use as-is
2. **Tier 2**: Scene too short → merge with neighbors
3. **Tier 3**: Scene too long → extract best sub-segment

**Suppression:** Once a scene is used, it's locked to prevent repetition.

In [15]:
@dataclass
class ClipCandidate:
    """A selected clip candidate ready for extraction."""
    target_duration: int
    start_time: float
    end_time: float
    actual_duration: float
    score: float
    source_scene_ids: List[int]
    selection_tier: int
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def select_clips(
    scenes: List[Scene],
    fused: FusedSignal,
    target_durations: List[int] = [15, 30, 60],
    duration_tolerance: float = 0.2
) -> List[ClipCandidate]:
    """
    Select best clips for each target duration using Refined Greedy Strategy.
    """
    print(f"\n🎯 Selecting clips for durations: {target_durations}")
    
    sorted_scenes = sorted(scenes, key=lambda s: s.score, reverse=True)
    locked_ids = set()
    candidates = []
    scores = fused.per_second_scores
    video_duration = fused.duration
    
    for target in target_durations:
        print(f"\n   🔍 Finding best {target}s clip...")
        
        if video_duration < target:
            print(f"      ⚠️ Video too short for {target}s clip")
            continue
        
        min_dur = target * (1 - duration_tolerance)
        max_dur = target * (1 + duration_tolerance)
        
        best_candidate = None
        
        for scene in sorted_scenes:
            if scene.id in locked_ids:
                continue
            
            # TIER 1: Scene fits perfectly
            if min_dur <= scene.duration <= max_dur:
                best_candidate = ClipCandidate(
                    target_duration=target,
                    start_time=scene.start_time,
                    end_time=scene.end_time,
                    actual_duration=scene.duration,
                    score=scene.score,
                    source_scene_ids=[scene.id],
                    selection_tier=1
                )
                print(f"      ✅ Tier 1: Scene {scene.id} fits ({scene.duration:.1f}s)")
                break
            
            # TIER 2: Scene too short → merge with neighbors
            if scene.duration < min_dur:
                scene_idx = next((i for i, s in enumerate(scenes) if s.id == scene.id), None)
                if scene_idx is not None:
                    merged_duration = scene.duration
                    merged_ids = [scene.id]
                    merged_end = scene.end_time
                    
                    for next_scene in scenes[scene_idx + 1:]:
                        if next_scene.id in locked_ids:
                            continue
                        merged_duration += next_scene.duration
                        merged_ids.append(next_scene.id)
                        merged_end = next_scene.end_time
                        
                        if merged_duration >= min_dur:
                            break
                    
                    if min_dur <= merged_duration <= max_dur:
                        start_sec = int(scene.start_time)
                        end_sec = min(int(merged_end), len(scores))
                        merged_score = np.mean(scores[start_sec:end_sec]) if end_sec > start_sec else 0
                        
                        best_candidate = ClipCandidate(
                            target_duration=target,
                            start_time=scene.start_time,
                            end_time=merged_end,
                            actual_duration=merged_duration,
                            score=merged_score,
                            source_scene_ids=merged_ids,
                            selection_tier=2
                        )
                        print(f"      ✅ Tier 2: Merged scenes {merged_ids} ({merged_duration:.1f}s)")
                        break
            
            # TIER 3: Scene too long → extract best sub-segment
            if scene.duration > max_dur:
                start_sec = int(scene.start_time)
                end_sec = int(scene.end_time)
                
                best_subscore = -1
                best_start = start_sec
                
                for win_start in range(start_sec, end_sec - target + 1):
                    win_end = min(win_start + target, len(scores))
                    if win_end <= win_start:
                        continue
                    win_score = np.mean(scores[win_start:win_end])
                    if win_score > best_subscore:
                        best_subscore = win_score
                        best_start = win_start
                
                if best_subscore >= 0:
                    best_candidate = ClipCandidate(
                        target_duration=target,
                        start_time=float(best_start),
                        end_time=float(best_start + target),
                        actual_duration=float(target),
                        score=best_subscore,
                        source_scene_ids=[scene.id],
                        selection_tier=3
                    )
                    print(f"      ✅ Tier 3: Sub-segment from scene {scene.id} at {best_start}s")
                    break
        
        if best_candidate:
            for sid in best_candidate.source_scene_ids:
                locked_ids.add(sid)
            candidates.append(best_candidate)
        else:
            print(f"      ⚠️ Could not find suitable {target}s clip")
    
    print(f"\n✅ Selected {len(candidates)} clips")
    return candidates


print("✅ select_clips() defined")

✅ select_clips() defined


In [16]:
# === Run Phase 5 ===
if 'fused_signal' in dir() and 'scenes' in dir():
    clip_candidates = select_clips(scenes, fused_signal)
    
    print("\n📋 CLIP CANDIDATES:")
    print("=" * 70)
    for c in clip_candidates:
        print(f"  {c.target_duration}s clip: {c.start_time:.1f}s - {c.end_time:.1f}s "
              f"(actual: {c.actual_duration:.1f}s, score: {c.score:.3f}, tier: {c.selection_tier})")
else:
    print("⚠️ Run Phase 4 first")


🎯 Selecting clips for durations: [15, 30, 60]

   🔍 Finding best 15s clip...
      ✅ Tier 2: Merged scenes [37, 38, 39, 40, 41] (12.9s)

   🔍 Finding best 30s clip...
      ✅ Tier 2: Merged scenes [32, 33, 34, 35, 36] (24.1s)

   🔍 Finding best 60s clip...
      ✅ Tier 2: Merged scenes [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23] (49.1s)

✅ Selected 3 clips

📋 CLIP CANDIDATES:
  15s clip: 167.9s - 180.8s (actual: 12.9s, score: 0.583, tier: 2)
  30s clip: 143.8s - 167.9s (actual: 24.1s, score: 0.346, tier: 2)
  60s clip: 61.4s - 110.5s (actual: 49.1s, score: 0.312, tier: 2)


---
## 🎬 PHASE 6: Production Layer (Clip Extraction)

**Quality Gates:**
1. Output file exists and size > 1KB
2. Duration matches **candidate.actual_duration** (not target) ±0.5s

In [17]:
@dataclass
class ExtractedClip:
    """Final extracted clip with validation results."""
    filename: str
    filepath: str
    target_duration: int
    expected_duration: float  # What we told FFmpeg to extract
    actual_duration: float    # What FFmpeg actually produced
    start_time: float
    end_time: float
    score: float
    filesize_mb: float
    passed_validation: bool
    validation_errors: List[str] = field(default_factory=list)
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def extract_clips(
    candidates: List[ClipCandidate],
    source_video: str,
    output_dir: str,
    duration_tolerance: float = 0.5
) -> List[ExtractedClip]:
    """
    Extract clips from source video using FFmpeg.
    
    FIX #1: Validates against candidate.actual_duration, not target_duration.
    """
    print(f"\n🎬 Extracting {len(candidates)} clips from source video...")
    
    source_path = Path(source_video)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    extracted = []
    
    for i, candidate in enumerate(candidates):
        clip_name = f"clip_{i+1:02d}_{candidate.target_duration}s.mp4"
        clip_path = output_path / clip_name
        
        print(f"\n   [{i+1}/{len(candidates)}] Extracting {clip_name}...")
        print(f"       Time: {candidate.start_time:.2f}s - {candidate.end_time:.2f}s ({candidate.actual_duration:.1f}s)")
        
        cmd = [
            "ffmpeg", "-y",
            "-ss", str(candidate.start_time),
            "-i", str(source_path),
            "-t", str(candidate.actual_duration),
            "-c:v", "libx264",
            "-c:a", "aac",
            "-preset", "medium",
            "-crf", "23",
            str(clip_path)
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        errors = []
        passed = True
        actual_duration = 0.0
        filesize_mb = 0.0
        
        if result.returncode != 0:
            errors.append(f"FFmpeg failed: {result.stderr[:200]}")
            passed = False
        elif not clip_path.exists():
            errors.append("Output file not created")
            passed = False
        else:
            # Quality Gate 1: File size
            filesize_mb = clip_path.stat().st_size / (1024 * 1024)
            if filesize_mb < 0.001:
                errors.append(f"File too small: {filesize_mb:.3f} MB")
                passed = False
            
            # Quality Gate 2: Duration check
            # ============================================================
            # FIX #1: Compare against candidate.actual_duration (what we
            # told FFmpeg), NOT target_duration (original goal)
            # ============================================================
            try:
                clip_meta = get_video_metadata(str(clip_path))
                actual_duration = clip_meta.duration
                
                # Compare to expected, not target
                dur_diff = abs(actual_duration - candidate.actual_duration)
                if dur_diff > duration_tolerance:
                    errors.append(
                        f"FFmpeg glitch: Output {actual_duration:.1f}s != "
                        f"Expected {candidate.actual_duration:.1f}s"
                    )
                    # This is a real problem - FFmpeg didn't do what we asked
                    passed = False
            except Exception as e:
                errors.append(f"Cannot read clip metadata: {e}")
                passed = False
        
        status = "✅" if passed else "❌"
        print(f"       {status} {clip_name}: {actual_duration:.1f}s, {filesize_mb:.2f} MB")
        if errors:
            for err in errors:
                print(f"          ⚠️ {err}")
        
        extracted.append(ExtractedClip(
            filename=clip_name,
            filepath=str(clip_path),
            target_duration=candidate.target_duration,
            expected_duration=candidate.actual_duration,
            actual_duration=actual_duration,
            start_time=candidate.start_time,
            end_time=candidate.end_time,
            score=candidate.score,
            filesize_mb=filesize_mb,
            passed_validation=passed,
            validation_errors=errors
        ))
    
    passed_count = sum(1 for c in extracted if c.passed_validation)
    print(f"\n✅ Extraction complete: {passed_count}/{len(extracted)} clips passed validation")
    
    return extracted


print("✅ extract_clips() defined (with FIX #1: correct duration validation)")

✅ extract_clips() defined (with FIX #1: correct duration validation)


In [18]:
# === Run Phase 6 ===
if 'clip_candidates' in dir() and clip_candidates and 'TEST_VIDEO_PATH' in dir():
    extracted_clips = extract_clips(
        candidates=clip_candidates,
        source_video=TEST_VIDEO_PATH,
        output_dir=str(CLIPS_DIR)
    )
else:
    print("⚠️ Run Phase 5 first")


🎬 Extracting 3 clips from source video...

   [1/3] Extracting clip_01_15s.mp4...
       Time: 167.90s - 180.80s (12.9s)
       ✅ clip_01_15s.mp4: 12.8s, 8.68 MB

   [2/3] Extracting clip_02_30s.mp4...
       Time: 143.80s - 167.90s (24.1s)
       ✅ clip_02_30s.mp4: 24.1s, 12.83 MB

   [3/3] Extracting clip_03_60s.mp4...
       Time: 61.40s - 110.50s (49.1s)
       ✅ clip_03_60s.mp4: 49.1s, 33.68 MB

✅ Extraction complete: 3/3 clips passed validation


In [19]:
# === Generate Final Output JSON ===
def generate_output_json(clips: List[ExtractedClip], output_dir: str) -> str:
    """Generate the final JSON output as specified in the FRD."""
    
    output = {
        "status": "completed",
        "total_clips": len(clips),
        "clips": []
    }
    
    for clip in clips:
        output["clips"].append({
            "filename": clip.filename,
            "duration": round(clip.actual_duration, 2),
            "target_duration": clip.target_duration,
            "expected_duration": round(clip.expected_duration, 2),
            "start_time": round(clip.start_time, 2),
            "end_time": round(clip.end_time, 2),
            "score": round(clip.score, 3),
            "filesize_mb": round(clip.filesize_mb, 2),
            "download_url": f"/clips/{clip.filename}",
            "passed_validation": clip.passed_validation
        })
    
    output_path = Path(output_dir) / "result.json"
    with open(output_path, "w") as f:
        json.dump(output, f, indent=2)
    
    print(f"\n💾 Output JSON saved to: {output_path}")
    return str(output_path)


if 'extracted_clips' in dir():
    result_json_path = generate_output_json(extracted_clips, str(OUTPUT_DIR))
    
    print("\n📄 FINAL OUTPUT:")
    print("=" * 60)
    with open(result_json_path) as f:
        print(json.dumps(json.load(f), indent=2))


💾 Output JSON saved to: D:\Freelancing_Work\ClipSpark\output\result.json

📄 FINAL OUTPUT:
{
  "status": "completed",
  "total_clips": 3,
  "clips": [
    {
      "filename": "clip_01_15s.mp4",
      "duration": 12.76,
      "target_duration": 15,
      "expected_duration": 12.9,
      "start_time": 167.9,
      "end_time": 180.8,
      "score": 0.583,
      "filesize_mb": 8.68,
      "download_url": "/clips/clip_01_15s.mp4",
      "passed_validation": true
    },
    {
      "filename": "clip_02_30s.mp4",
      "duration": 24.1,
      "target_duration": 30,
      "expected_duration": 24.1,
      "start_time": 143.8,
      "end_time": 167.9,
      "score": 0.346,
      "filesize_mb": 12.83,
      "download_url": "/clips/clip_02_30s.mp4",
      "passed_validation": true
    },
    {
      "filename": "clip_03_60s.mp4",
      "duration": 49.1,
      "target_duration": 60,
      "expected_duration": 49.1,
      "start_time": 61.4,
      "end_time": 110.5,
      "score": 0.312,
      "file

---
## ✅ PIPELINE COMPLETE!

### Applied Fixes:
1. ✅ **Fix #1**: Duration validation compares `actual_duration` vs `expected_duration` (not target)
2. ✅ **Fix #2**: Signal weights adjusted to 0.6 Motion / 0.4 Human for "Spark" branding
3. ✅ **Fix #3**: Docker-ready paths using `os.getenv("APP_DIR", "...")`

### Output Files:
- `output/clips/clip_01_15s.mp4`
- `output/clips/clip_02_30s.mp4`
- `output/clips/clip_03_60s.mp4`
- `output/result.json`

In [20]:
# === Final Summary ===
print("\n" + "=" * 70)
print("🎉 CLIPSPARK ENGINE - COMPLETE PIPELINE SUMMARY")
print("=" * 70)

if 'validation' in dir() and validation.metadata:
    print(f"\n📹 Source Video: {Path(validation.metadata.filepath).name}")
    print(f"   Duration: {validation.metadata.duration:.1f}s")
    print(f"   Resolution: {validation.metadata.width}x{validation.metadata.height}")

if 'scenes' in dir():
    print(f"\n🎬 Scenes: {len(scenes)} detected")

if 'fused_signal' in dir():
    print(f"\n🧮 Fusion Weights: Motion={fused_signal.motion_weight}, Human={fused_signal.human_weight}")

if 'motion_signal' in dir() and 'human_signal' in dir():
    print(f"\n🧠 Signals:")
    print(f"   Motion: mean={motion_signal.smoothed_scores.mean():.3f}")
    print(f"   Human: mean={human_signal.smoothed_scores.mean():.3f}")

if 'extracted_clips' in dir():
    print(f"\n📼 Extracted Clips:")
    for clip in extracted_clips:
        status = "✅" if clip.passed_validation else "❌"
        print(f"   {status} {clip.filename}: {clip.actual_duration:.1f}s (target: {clip.target_duration}s), score={clip.score:.3f}")

print(f"\n📂 Output Directory: {CLIPS_DIR}")
print("=" * 70)


🎉 CLIPSPARK ENGINE - COMPLETE PIPELINE SUMMARY

📹 Source Video: testing.mp4
   Duration: 180.7s
   Resolution: 1920x1080

🎬 Scenes: 41 detected

🧮 Fusion Weights: Motion=0.6, Human=0.4

🧠 Signals:
   Motion: mean=0.135
   Human: mean=0.236

📼 Extracted Clips:
   ✅ clip_01_15s.mp4: 12.8s (target: 15s), score=0.583
   ✅ clip_02_30s.mp4: 24.1s (target: 30s), score=0.346
   ✅ clip_03_60s.mp4: 49.1s (target: 60s), score=0.312

📂 Output Directory: D:\Freelancing_Work\ClipSpark\output\clips
